<a href="https://colab.research.google.com/github/crialejo24/DOWNSCALING/blob/main/Dataset_S2_LS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook requiere: (1) una cuenta de Google Earth Engine propia con un proyecto creado en https://developers.google.com/earth-engine, y (2) reemplazar project='...' por tu propio project ID. El dataset ya generado está disponible en Hugging Face: https://huggingface.co/datasets/Crialejo924/DOWNSCALING/tree/main

In [2]:
from google.colab import drive
import ee
import geemap
import pandas as pd
import numpy as np
import random
import cv2
from google.colab.patches import cv2_imshow
import os

#1. USE COLAB WITH A GOOGLE PERSONAL ACCOUNT (NOT @elpoli.edu.co)
#2. SIGN IN TO https://developers.google.com/earth-engine
#3. CREATE A PROJECT, IN THIS CASE I NAMED IT "rubenchov". USE YOUR OWN.
!rm -rf /content/DOWNSCALING
!git clone https://github.com/crialejo24/DOWNSCALING.git /content/DOWNSCALING
%cd /content/DOWNSCALING
!pip install -q -r requirements.txt

from src.gee_utils import cloud_percentage_s2, water_percentage_s2, cloud_percentage_landsat, variability_s2
from src.image_utils import normalize_if_needed, to_uint8_vis, to_uint8_rgb
from src.dataset_builder import get_temporal_pair_rgb, save_images, nombre_random
# Drive
drive.mount('/content/drive')

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='vamos-486322')  #CRISTIAN
#ee.Initialize(project='andresc-488619') #GUSTAVO

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/DOWNSCALING'...
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/DOWNSCALING'
/content/DOWNSCALING
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


MessageError: Error: credential propagation was unsuccessful

###GENERACIÓN DEL DATASET

In [ ]:
import random
import datetime
import csv
import os
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# =====================================
# CONFIGURACIÓN
# =====================================
tipo='train'
#tipo='test'

if tipo=='test':
    base_folder = "/content/drive/MyDrive/DOWNSCALING_2/test_GeoR/"
else:
    base_folder = "/content/drive/MyDrive/DOWNSCALING_2/train_GeoR/"

os.makedirs(base_folder, exist_ok=True)

csv_filename = os.path.join(base_folder, "dataset_metadata_192_48_64.csv")

geolocator = Nominatim(user_agent="dataset_generator")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

TARGET_IMAGES = 10000
S2_RESIZED_DIM = 192
LS_RESIZED_DIM_4 = S2_RESIZED_DIM // 4  # LR/4
LS_RESIZED_DIM_3 = S2_RESIZED_DIM // 3  # LR/3
max_intentos = 20000

# =====================================
# FUNCIÓN FECHA ALEATORIA
# =====================================
def random_month_between(start_year=2022, end_year=2023, end_month_limit=1):
    year = random.randint(start_year, end_year)
    month = random.randint(1, 12) if year != end_year else random.randint(1, end_month_limit)
    start_date = datetime.date(year, month, 1)
    end_date = datetime.date(year + (month == 12), 1 if month == 12 else month + 1, 1) - datetime.timedelta(days=1)
    return str(start_date), str(end_date)

# =====================================
# CONTAR EXISTENTES
# =====================================
existing_names = set()
imagenes_generadas = 0

if os.path.isfile(csv_filename):
    with open(csv_filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader, None)
        for row in reader:
            if len(row) > 0:
                existing_names.add(row[0])
                imagenes_generadas += 1

print(f"📊 Imágenes ya existentes: {imagenes_generadas}")

# =====================================
# CREAR CSV SI NO EXISTE
# =====================================
file_exists = os.path.isfile(csv_filename)

with open(csv_filename, mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    if not file_exists:
        writer.writerow([
            "filename",
            "latitude",
            "longitude",
            "sentinel_original_dimension",
            "landsat_original_dimension",
            "sentinel_resized_dimension",
            "landsat_resized_LR4_dimension",
            "landsat_resized_LR3_dimension",
            "city",
            "country",
            "sentinel_date",
            "landsat_date"
        ])

    intentos = 0

    while imagenes_generadas < TARGET_IMAGES and intentos < max_intentos:
        intentos += 1
        lat = random.uniform(-30, 30)
        lon = random.uniform(-179, 179)
        start_date, end_date = random_month_between()

        print(f"\nIntento {intentos}")
        print(f"📍 {lat:.4f}, {lon:.4f}")
        print(f"📅 {start_date} → {end_date}")

        result = get_temporal_pair_rgb(
            start=start_date,
            end=end_date,
            latitude=lat,
            longitude=lon,
            dim=S2_RESIZED_DIM,
            max_cloud_pct=1,
            day_tolerance=5
        )

        if result is None:
            print("❌ No válida")
            continue

        # =========================
        # Nombre único
        # =========================
        nombre = nombre_random()
        while nombre in existing_names:
            nombre = nombre_random()

        # =========================
        # Guardar imágenes HR/LR
        # =========================
        save_images(result, nombre, base_folder, dim=S2_RESIZED_DIM)

        # =========================
        # Obtener ciudad y país
        # =========================
        try:
            location = reverse((lat, lon), language='en')
            if location and 'address' in location.raw:
                address = location.raw['address']
                city = address.get('city', address.get('town', address.get('village', 'Unknown')))
                country = address.get('country', 'Unknown')
            else:
                city = "Unknown"
                country = "Unknown"
        except:
            city = "Unknown"
            country = "Unknown"

        # =========================
        # METADATOS
        # =========================
        sentinel_dim_original = result["s2_dimension"]
        landsat_dim_original = result["landsat_dimension"]

        sentinel_dim_resized = (S2_RESIZED_DIM, S2_RESIZED_DIM)
        landsat_dim_resized_LR4 = (LS_RESIZED_DIM_4, LS_RESIZED_DIM_4)
        landsat_dim_resized_LR3 = (LS_RESIZED_DIM_3, LS_RESIZED_DIM_3)

        sentinel_date = result["s2_date"]
        landsat_date = result["landsat_date"]

        # =========================
        # Guardar CSV
        # =========================
        writer.writerow([
            nombre,
            lat,
            lon,
            sentinel_dim_original,
            landsat_dim_original,
            sentinel_dim_resized,
            landsat_dim_resized_LR4,
            landsat_dim_resized_LR3,
            city,
            country,
            sentinel_date,
            landsat_date
        ])

        file.flush()
        existing_names.add(nombre)
        imagenes_generadas += 1

        print(f"✅ Imagen {imagenes_generadas}/{TARGET_IMAGES} guardada")

print("\n=================================")
print(f"Total final: {imagenes_generadas}")
print("Proceso terminado")



##VERIFICACIÓN DE CORRESPONDENCIA DE IMAGENES ENTRE LAS CARPETAS

In [ ]:
import os
import pandas as pd

# ===============================
# RUTA BASE
# ===============================
tipo='test'
#tipo='train'

if tipo == 'test':
    base = "/content/drive/MyDrive/DOWNSCALING_2/test_GeoR/"
else:
    base = "/content/drive/MyDrive/DOWNSCALING_2/train_GeoR/"

# Ruta del CSV
csv_path = os.path.join(base, "dataset_metadata_192_48_64.csv")  # ajusta el nombre

# ===============================
# LEER CSV
# ===============================
df = pd.read_csv(csv_path)

# Crear columna auxiliar con clave
df['clave'] = df['filename'].str[:10]

claves_csv = set(df['clave'])

print("Claves en CSV:", len(claves_csv))

# ===============================
# CARPETAS
# ===============================
carpetas = [
    "HR_192_norm/",
    "LR_48_normx4/",
    "FB_HR_192/",
    "FB_LR_48/",
    "HR_192_vis/",
    "LR_48_visx4/",
    "HR_192_mod/",
    "LR_48_modx4/",
    "LR_64_normx3/",
    "LR_64_modx3/",
    "LR_64_visx3/"
]

paths = [os.path.join(base, c) for c in carpetas]

# ===============================
# LEER ARCHIVOS
# ===============================
archivos_por_carpeta = {}

for path in paths:
    archivos = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]

    claves = {}
    for a in archivos:
        clave = a[:10]
        claves[clave] = a

    archivos_por_carpeta[path] = claves

# ===============================
# INTERSECCIÓN DE CLAVES
# ===============================
sets = [set(d.keys()) for d in archivos_por_carpeta.values()]
claves_comunes = set.intersection(*sets)

# Intersección con CSV
claves_comunes = claves_comunes.intersection(claves_csv)

print("Imágenes válidas (carpetas + CSV):", len(claves_comunes))

# ===============================
# BORRAR ARCHIVOS SOBRANTES
# ===============================
eliminadas = 0

for path, archivos_dict in archivos_por_carpeta.items():
    for clave, archivo in archivos_dict.items():
        if clave not in claves_comunes:
            ruta = os.path.join(path, archivo)
            os.remove(ruta)
            eliminadas += 1
            print("Eliminado archivo:", ruta)

print("Total archivos eliminados:", eliminadas)

# ===============================
# LIMPIAR CSV
# ===============================
df_filtrado = df[df['clave'].isin(claves_comunes)].copy()

eliminadas_csv = len(df) - len(df_filtrado)

# Opcional: quitar columna auxiliar
df_filtrado = df_filtrado.drop(columns=['clave'])

# Guardar (sobrescribe)
df_filtrado.to_csv(csv_path, index=False)

print("Filas eliminadas del CSV:", eliminadas_csv)
print("CSV alineado correctamente.")

print("Dataset completamente alineado (carpetas + CSV).")